# SpendShield First Research-Only Baseline

This notebook evaluates the first transparent baseline on the isolated synthetic feature matrix.

## Objective

Compare a majority-class reference with deterministic Gaussian Naive Bayes using temporal train, validation, and test partitions.

## Non-goals

This is not fraud detection, a risk-scoring API, production inference, automatic blocking, notification, or a real-world performance evaluation.

## Target definition

The target is `scenario_label`, exposed as `target_scenario_label`. It represents intentionally generated synthetic scenarios, not confirmed fraud, actual fraud, or a real-world risk label. Only multiclass evaluation is selected.

In [1]:
from pathlib import Path
import sys
REPOSITORY_ROOT = next(candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / "ml" / "synthetic_dataset_generator.py").exists())
sys.path.insert(0, str(REPOSITORY_ROOT))
SOURCE_DIR = REPOSITORY_ROOT / "data" / "synthetic"
FEATURE_DIR = SOURCE_DIR / "features"
import json
from ml.research_baseline import evaluate_research_baseline
RESULT_PATH = SOURCE_DIR / 'baseline' / 'research_baseline_results.json'
results = evaluate_research_baseline(FEATURE_DIR, RESULT_PATH)
{'dataset_type': results['dataset_type'], 'dataset_version': results['dataset_version'], 'feature_count': results['feature_count'], 'target': results['target']}

{'dataset_type': 'synthetic_research',
 'dataset_version': 'v1',
 'feature_count': 28,
 'target': {'classes': ['normal',
   'synthetic_behavior_deviation',
   'synthetic_combined_pattern',
   'synthetic_high_amount',
   'synthetic_rapid_repeat',
   'synthetic_unusual_time'],
  'column': 'target_scenario_label',
  'notice': 'Classification of intentionally generated synthetic scenarios, not detection of real fraud.',
  'real_fraud': False,
  'source_field': 'scenario_label',
  'type': 'multiclass synthetic scenario target'}}

## Evaluation method

Gaussian Naive Bayes parameters are fit on train rows only. Validation is reported without test tuning. Test is evaluated once as the final held-out temporal result. Preprocessing was fit from training data only; no random row split is used.

In [2]:
summary = {'data_usage': results['data_usage'], 'class_counts': {model_name: {split_name: split_result['class_counts'] for split_name, split_result in model_result.items() if split_name in {'validation', 'test'}} for model_name, model_result in results['models'].items()}}
summary

{'data_usage': {'fit': 'train only',
  'validation': 'reported for research comparison; no test tuning',
  'test': 'final held-out evaluation after fixed baseline configuration',
  'random_row_split': False},
 'class_counts': {'majority_class': {'validation': {'normal': 1124,
    'synthetic_behavior_deviation': 37,
    'synthetic_combined_pattern': 27,
    'synthetic_high_amount': 117,
    'synthetic_rapid_repeat': 89,
    'synthetic_unusual_time': 87},
   'test': {'normal': 1132,
    'synthetic_behavior_deviation': 36,
    'synthetic_combined_pattern': 19,
    'synthetic_high_amount': 114,
    'synthetic_rapid_repeat': 52,
    'synthetic_unusual_time': 105}},
  'gaussian_naive_bayes': {'validation': {'normal': 1124,
    'synthetic_behavior_deviation': 37,
    'synthetic_combined_pattern': 27,
    'synthetic_high_amount': 117,
    'synthetic_rapid_repeat': 89,
    'synthetic_unusual_time': 87},
   'test': {'normal': 1132,
    'synthetic_behavior_deviation': 36,
    'synthetic_combined_

In [3]:
metrics = {model_name: {split_name: split_result['metrics'] for split_name, split_result in model_result.items() if split_name in {'validation', 'test'}} for model_name, model_result in results['models'].items()}
metrics

{'majority_class': {'validation': {'accuracy': 0.758947,
   'macro_precision': 0.126491,
   'macro_recall': 0.166667,
   'macro_f1': 0.143826,
   'weighted_f1': 0.654938,
   'classes': ['normal',
    'synthetic_behavior_deviation',
    'synthetic_combined_pattern',
    'synthetic_high_amount',
    'synthetic_rapid_repeat',
    'synthetic_unusual_time'],
   'per_class': {'normal': {'precision': 0.758947,
     'recall': 1.0,
     'f1': 0.862956,
     'support': 1124},
    'synthetic_behavior_deviation': {'precision': 0.0,
     'recall': 0.0,
     'f1': 0.0,
     'support': 37},
    'synthetic_combined_pattern': {'precision': 0.0,
     'recall': 0.0,
     'f1': 0.0,
     'support': 27},
    'synthetic_high_amount': {'precision': 0.0,
     'recall': 0.0,
     'f1': 0.0,
     'support': 117},
    'synthetic_rapid_repeat': {'precision': 0.0,
     'recall': 0.0,
     'f1': 0.0,
     'support': 89},
    'synthetic_unusual_time': {'precision': 0.0,
     'recall': 0.0,
     'f1': 0.0,
     'supp

## Interpretation

Metrics measure classification performance on this generator's synthetic scenarios only. They are not fraud recall, prevention rate, financial savings, risk reduction, or production performance. High synthetic performance may indicate a generator shortcut.

In [4]:
first = evaluate_research_baseline(FEATURE_DIR, output_path=None)
second = evaluate_research_baseline(FEATURE_DIR, output_path=None)
reproducibility = {'same_model_results': first['models'] == second['models'], 'test_is_held_out': first['data_usage']['test'].startswith('final held-out'), 'production_inference_created': results['production_inference_created'], 'model_artifact_written': results['model_artifact_written']}
assert reproducibility['same_model_results']
reproducibility

{'same_model_results': True,
 'test_is_held_out': True,
 'production_inference_created': False,
 'model_artifact_written': False}

## Conclusion and limitations

The majority baseline provides the comparison floor. Gaussian Naive Bayes is a simple interpretable research baseline. Results apply only to synthetic scenario labels. No model artifact, inference route, fraud API, blocking, notification, or production decision has been created.